# 03 大模型应用开发 · 第01课：调用真实大模型API

> 你已经写过一个FastAPI聊天后端，但其中的AI回复是固定规则生成的。本课只做一件事：理解并完成一次真实的文本模型调用。多轮对话、流式响应、RAG和Agent以后再学。

## 1. 学完后你能独立完成什么

你将能够：

1. 用自己的话区分ChatGPT、模型、API和Python SDK；
2. 把API密钥安全地放在本地`.env`，而不是代码中；
3. 逐行解释一次最小的Responses API调用；
4. 从响应对象取得模型生成的文本；
5. 区分本地配置错误、认证错误、额度错误和网络错误；
6. 完成三个不联网的基础函数，为下一课接回聊天后端做准备。

本课依赖你已经学过的：模块导入、函数、字符串、字典、环境变量、异常和pytest。不要求你先了解大模型内部原理。

## 2. 运行环境准备

Notebook右上角应选择`Python (.venv 学习)`或指向仓库`.venv/bin/python`的内核。依赖已经安装时无需重复安装；需要重新安装可在本课程目录运行：

```powershell
python -m pip install -r requirements.txt
```

`python`启动当前解释器；`-m pip`让同一个解释器运行pip；`install`表示安装；`-r requirements.txt`表示从文件读取依赖。`openai`是官方Python SDK，`python-dotenv`读取本地`.env`，`pytest`运行作业测试。

先运行下面的环境检查。它只检查包，不访问API。

In [ ]:
import sys
import openai
import dotenv

print("Python解释器：", sys.executable)
print("OpenAI SDK版本：", openai.__version__)
print("✅ 依赖导入成功；此单元没有访问网络。")

## 3. ChatGPT、模型、API和SDK分别是什么

先用餐厅类比：

| 名称 | 餐厅类比 | 在本课中的含义 |
|---|---|---|
| ChatGPT | 已装修好、供顾客使用的餐厅 | 人直接使用的完整产品，有界面和产品功能 |
| 模型 | 厨师 | 接收输入并生成输出的核心能力 |
| API | 点餐窗口和点餐规则 | 程序按照约定格式发送请求的入口 |
| SDK | 帮你填写订单的服务员 | 把HTTP细节包装成Python类和方法 |

ChatGPT订阅和API账户不是同一项服务。使用API可能产生单独费用，能否调用某个模型取决于API账户的权限和额度。

一次调用的数据流：

```text
你的Python代码
  ↓ 传入模型名和问题
OpenAI Python SDK
  ↓ 生成HTTPS请求
OpenAI API服务
  ↓ 验证密钥并调用模型
响应对象返回Python程序
  ↓ 读取output_text
得到回答字符串
```

Python负责执行你的代码；SDK负责组织网络请求和解析响应；远端API负责认证和运行模型。用户代码中没有一行手写JSON转响应对象，这一步由SDK自动完成。

### 课堂练习1：判断职责

**目标**：区分SDK和远端API。

**题目**：把`request_builder`和`model_runner`分别填成`"SDK"`或`"远端API"`。前者负责把Python参数组织成请求，后者负责真正生成回答。

**预期结果**：检测单元输出✅。本练习不访问网络。

In [ ]:
request_builder = "TODO"
model_runner = "TODO"

In [ ]:
try:
    if request_builder == "TODO" or model_runner == "TODO":
        print("❌ 还没有填写两个变量。")
    elif request_builder != "SDK":
        print("❌ 请回看：谁把Python参数组织成网络请求？")
    elif model_runner != "远端API":
        print("❌ 请回看：真正运行模型的是本地SDK还是远端服务？")
    else:
        print("✅ SDK与远端API的职责区分正确。")
except NameError:
    print("❌ 请先运行答题单元。")

<details><summary>完成练习1后查看参考答案</summary>

`request_builder = "SDK"`，`model_runner = "远端API"`。SDK只是本地工具，真正的模型不在你的Notebook进程中运行。
</details>

## 4. API密钥为什么不能写进代码

API密钥是程序访问API的凭证，类似密码。服务端根据它识别API账户并记录用量。以下写法危险：

```python
client = OpenAI(api_key="这里写真实密钥")  # 不要这样做
```

Notebook会保存代码，Git会保存历史；即使之后删掉这一行，密钥也可能仍在旧提交中。正确方式是把真实值放进本地`.env`：

```text
OPENAI_API_KEY=你的真实API密钥
OPENAI_MODEL=你的账户可用模型ID
```

课程提供的是`.env.example`，它只展示字段名，不包含真实密钥。复制为`.env`后再填写；仓库`.gitignore`会忽略`.env`。模型名不是秘密，但做成配置后可以不改代码就切换。

## 5. 先读取配置，再创建客户端

下面只读取配置，不调用API。

- `load_dotenv`来自`python-dotenv`，是函数；`load_dotenv()`从当前目录及其父目录附近查找`.env`，把键值加载到当前Python进程，返回`bool`表示是否找到文件。
- `os.getenv`来自Python标准库`os`；`os.getenv("名称")`读取环境变量，找不到时返回`None`。
- `None`表示没有值，不是空字符串。空字符串`""`表示变量存在但内容为空，两种情况都不能作为有效配置。

正常：`.env`存在且两项有值。变化：也可以在操作系统中预先设置环境变量，此时不依赖`.env`。错误：缺少密钥或模型名时，应在网络请求前停止。

In [ ]:
import os
from dotenv import load_dotenv

dotenv_found = load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_MODEL")

print("找到.env文件：", dotenv_found)
print("已配置API密钥：", isinstance(api_key, str) and bool(api_key.strip()))
print("模型名称：", model_name if model_name else "未配置")
# 永远不要print(api_key)，这里只打印是否存在。

### 课堂练习2：识别缺失和空配置

**目标**：理解`None`、空字符串和有效字符串的区别。

**题目**：完成`has_text(value)`。只有`value`是字符串，并且去除首尾空白后仍有内容时才返回`True`。

**限制**：只使用刚刚示例中的`isinstance`、`strip`和布尔判断。

**预期结果**：`None`、`""`和`"   "`为False，`"demo-model"`为True。

In [ ]:
def has_text(value):
    # TODO：返回一个布尔值
    return False

In [ ]:
try:
    _check_cases = [(None, False), ("", False), ("   ", False), (" demo-model ", True)]
    _check_wrong = [(value, has_text(value), expected) for value, expected in _check_cases if has_text(value) is not expected]
    if _check_wrong:
        print("❌ 这些输入判断不正确：", _check_wrong)
    else:
        print("✅ 缺失、空白和有效配置均判断正确。")
except NameError:
    print("❌ 请先定义并运行has_text。")
except Exception as _check_error:
    print(f"❌ 函数执行出错：{_check_error}")

<details><summary>完成练习2后查看参考答案</summary>

`return isinstance(value, str) and bool(value.strip())`。`and`从左到右判断，非字符串时不会继续调用`strip()`。
</details>

## 6. 创建OpenAI客户端

`OpenAI`来自官方`openai`包，是一个类。类可以看作创建对象的模板；`OpenAI()`调用这个类，返回一个客户端对象。客户端保存连接API所需的配置，并提供`responses`等功能入口。

```python
from openai import OpenAI
client = OpenAI()
```

这里省略了`api_key`关键字参数。SDK会自动读取`OPENAI_API_KEY`；用户代码中没有对应读取行，这是`OpenAI()`内部行为。若环境变量缺失，创建客户端或请求时会报认证配置错误。

显式写`OpenAI(api_key=api_key)`在技术上可行，但前提是`api_key`来自安全配置；绝不能把真实密钥字面量写进代码。创建客户端本身不等于已经调用模型，真正的网络请求发生在下一节调用`create()`时。

In [ ]:
from openai import OpenAI

client = None
if isinstance(api_key, str) and api_key.strip():
    client = OpenAI()
    print("✅ 客户端对象已创建；还没有发出模型请求。")
else:
    print("⚠️ 尚未配置OPENAI_API_KEY，先完成.env再运行真实请求。")

## 7. 发出一次最小文本请求

核心调用是：

```python
response = client.responses.create(
    model=model_name,
    input=question,
)
```

逐项解释：

| 部分 | 类型与来源 | 含义 |
|---|---|---|
| `client.responses` | 客户端对象提供的功能入口 | 管理Responses API调用 |
| `create` | SDK方法 | 创建一次模型响应；调用时发生网络请求 |
| `model` | 必填关键字参数，字符串，由开发者提供 | 选择API账户可用的模型ID |
| `input` | 必填关键字参数，本课使用字符串，由用户问题产生 | 发送给模型的文本 |
| 返回值 | SDK响应对象 | 包含ID、状态、输出项、用量等信息 |

`model_name`不能为`None`或空字符串；`question`也应先去除首尾空白并拒绝空问题。调用可能产生API费用。下面的单元只有在客户端和模型名都准备好时才发送一次请求，否则只给提示。

In [ ]:
question = "请用一句话解释：API为什么适合让程序调用模型？"
clean_question = question.strip()

response = None
if client is None:
    print("⚠️ 客户端未创建：请先配置OPENAI_API_KEY并重新运行第5、6节。")
elif not isinstance(model_name, str) or not model_name.strip():
    print("⚠️ 未配置OPENAI_MODEL，请在.env中填写账户可用的模型ID。")
else:
    response = client.responses.create(
        model=model_name.strip(),
        input=clean_question,
    )
    print("✅ 请求完成，响应对象类型：", type(response).__name__)

## 8. 从响应对象取得文本

`response`不是字符串，而是SDK响应对象。Responses API的底层`output`可能包含多种输出项，不只文本。SDK提供`response.output_text`便捷属性，把其中的文本内容汇总成字符串。用户代码中没有遍历底层输出项的循环，因为SDK替你完成了汇总。

正常情况：`output_text`是非空字符串。变化情况：回答内容会随问题和模型变化，不能用逐字相等判断真实模型质量。边界情况：若没有可用文本，业务代码应明确报错，不能把空字符串当作成功回答。

In [ ]:
if response is None:
    print("⚠️ 还没有真实响应，请先完成并运行第7节。")
else:
    answer = response.output_text
    print("模型回答：")
    print(answer)

### 课堂练习3：检查响应文本

**目标**：练习读取对象属性、清理文本和处理空值。

**题目**：完成`get_text(result)`。读取`result.output_text`；只有它是字符串且清理后非空时才返回，否则抛出`ValueError("响应没有文本")`。

**限制**：不访问网络；检测单元会提供简单对象。

**预期结果**：正常文本被清理，空文本和缺少属性都会被拒绝。

In [ ]:
def get_text(result):
    # TODO：读取、检查、清理并返回output_text
    raise NotImplementedError

In [ ]:
class _CheckResult:
    def __init__(self, output_text):
        self.output_text = output_text

try:
    _check_normal = get_text(_CheckResult("  回答  "))
    if _check_normal != "回答":
        print("❌ 正常文本需要去除首尾空白后返回。")
    else:
        _check_rejected = 0
        for _check_value in ("   ", None):
            try:
                get_text(_CheckResult(_check_value))
            except ValueError:
                _check_rejected += 1
        try:
            get_text(object())
        except ValueError:
            _check_rejected += 1
        if _check_rejected == 3:
            print("✅ 正常、空值和缺少属性三类情况均处理正确。")
        else:
            print("❌ 空文本、None和缺少属性都应抛出ValueError。")
except NameError:
    print("❌ 请先定义并运行get_text。")
except NotImplementedError:
    print("❌ get_text尚未完成。")
except Exception as _check_error:
    print(f"❌ 函数出现未预期错误：{_check_error}")

<details><summary>完成练习3后查看参考答案</summary>

先用`getattr(result, "output_text", None)`读取属性；确认它是字符串，再`strip()`；结果为空则抛`ValueError`，否则返回清理后的文本。
</details>

## 9. 常见错误按层排查

| 发生阶段 | 常见现象 | 首先检查 |
|---|---|---|
| 创建客户端前 | `.env`找不到、变量为`None` | 文件名、课程目录、变量名拼写 |
| 认证 | API拒绝凭证 | 密钥是否属于API账户、是否完整、是否已撤销 |
| 模型选择 | 模型不存在或无权限 | `OPENAI_MODEL`是否为账户可用的准确ID |
| 额度与限流 | 请求过多或账户没有可用额度 | API账户用量、额度和速率限制 |
| 网络 | 连接超时或无法连接 | 网络、代理和服务连通性 |
| 响应处理 | 对象存在但业务没有文本 | `output_text`是否为非空字符串 |

不要用一个宽泛的`except Exception: return "失败"`隐藏所有原因。调试时保留异常类型和信息，但日志中绝不能打印API密钥。

## 10. 完整执行流程回顾

```text
1. 你在本地.env配置密钥和模型名
2. load_dotenv()把配置加载到当前Python进程
3. os.getenv()读取配置；你的代码先检查缺失和空值
4. OpenAI()创建SDK客户端，但此时尚未调用模型
5. 你的代码清理并检查用户问题
6. client.responses.create(model=..., input=...)发出网络请求
7. 远端服务认证、运行模型并返回结果
8. SDK把网络响应解析成Python响应对象
9. 你的代码读取并检查response.output_text
```

你负责配置、输入、参数、错误处理和业务使用；SDK负责请求与解析；远端API负责认证和运行模型。

## 11. 课末作业

只修改`ai_reply.py`中的三个TODO；不要修改测试。作业不调用真实API。

### 函数1：`normalize_question(question)`

- 输入必须是`str`，否则抛出包含“字符串”的`TypeError`；
- 去除首尾空白；
- 清理后为空则抛出包含“问题”的`ValueError`；
- 成功时返回清理后的字符串。

### 函数2：`read_model_name(environ)`

- `environ`是类似`os.environ`的字符串映射；
- 读取`OPENAI_MODEL`；
- 缺失、空字符串或只有空白时，抛出包含`OPENAI_MODEL`的`ValueError`；
- 成功时返回清理后的模型名。

### 函数3：`extract_output_text(response)`

- 读取`response.output_text`；
- 属性缺失、值不是字符串或清理后为空时，抛出包含“文本”的`ValueError`；
- 成功时返回清理后的文本。

运行：

```powershell
python -m pytest -q
```

应收集并通过13项测试。测试只验证本地Python行为，不读取`.env`、不访问网络、不产生API费用。

## 12. 理解题

1. ChatGPT、模型、API和SDK分别承担什么角色？
2. 一次模型调用中，哪些工作由你的代码、SDK和远端API分别完成？
3. 为什么真实API密钥不能写进Notebook？`.env.example`为什么可以提交？
4. `load_dotenv()`和`os.getenv()`分别做什么？变量缺失时`os.getenv()`返回什么？
5. `OpenAI()`何时读取密钥？创建客户端是否已经产生模型调用？
6. `client.responses.create()`中的`model`、`input`和返回值分别是什么？
7. 为什么不能假设响应对象本身就是字符串？`output_text`有什么作用？
8. 认证失败、模型无权限、额度不足和网络超时为什么要分开排查？

## 13. 小结与官方资料

本课只建立了一条最小而完整的链路：安全配置 → 创建客户端 → 发送一个问题 → 得到响应对象 → 提取文本 → 按错误层级排查。下一课才讨论如何设计提示词和控制模型行为。

课程依据当前[OpenAI模型目录](https://developers.openai.com/api/docs/models)和[模型与Responses API指导](https://developers.openai.com/api/docs/guides/latest-model)编写。模型可用性、价格和账户权限会变化，`OPENAI_MODEL`应填写你的API账户实际可用的模型ID。